In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
%load_ext aiida
%aiida
import urllib.parse

import ipywidgets as ipw
import matplotlib.pyplot as plt
import numpy as np
from aiida import orm
from IPython.display import clear_output, display

from surfaces_tools import helpers, utils
import html
from surfaces_tools.utils.calculation_model import (
    structure_for_display,
    neb_status_for_view,
    neb_status_html,
)

# Nudged Elastic Band calculations

In [ ]:
all_ase = []

In [ ]:
def make_replica_html(structure_data_list, energies, distances, replica_indices=None):
    html = "<table>"

    n_col = 4
    if replica_indices is None:
        replica_indices = range(len(structure_data_list))
    for i, (rep, index) in enumerate(zip(structure_data_list, replica_indices)):
        en, dist = energies[index], distances[index]
        thumbnail = rep.get_extra("thumbnail", None) or utils.ase_to_thumbnail(
            structure_for_display(rep)[0]
        )
        # The table cell.
        if i % n_col == 0:
            html += "<tr>"
        html += (
            f'<td><img width="400px" src="data:image/png;base64,{thumbnail}" title="">'
        )

        # Output some information about the replica.
        html += f"<p><b>Nr: </b>{index} <br> <b>Energy:</b> {en:.6f} eV <br> <b>Dist. to prev:</b> {dist:.4f} ang</p>"
        html += f"<p>pk: {rep.pk}</p>"
        # And the download link.
        html += f'<p><a target="_blank" href="export_structure.ipynb?uuid={rep.uuid}">View & export</a></p><td>'
        if i % n_col == n_col - 1:
            html += "</tr>"

    html += "</tr>"
    html += "</table>"
    return html

In [ ]:
def sorted_opt_rep_keys(keys):
    return sorted(
        [(int(key.split("_")[2]), key) for key in keys if "opt_replica" in key]
    )


def process_and_show_neb(_=None):
    global all_ase
    all_ase = []

    wc = orm.load_node(pk_select.value)
    structure_data_list = []
    replica_indices = []
    status = neb_status_for_view(wc)
    btn_show.disabled = True
    with main_out:
        clear_output()

    try:
        # old workchain
        old_workchain = True
        nreplicas = wc.inputs["nreplicas"].value
    except Exception:
        # new workchain
        old_workchain = False
        nreplicas = wc.inputs.neb_params["number_of_replica"]

    for i_rep in range(nreplicas):
        if old_workchain:
            label = f"opt_replica_{i_rep}"
        else:
            label = f"opt_replica_{str(i_rep).zfill(3)}"

        if label in wc.outputs:
            structure_data_list.append(wc.outputs[label])
            replica_indices.append(i_rep)

    with main_out:
        display(ipw.HTML(neb_status_html(status)))
        if len(structure_data_list) != nreplicas:
            print(
                "Some saved replica geometries are unavailable; available energies remain viewable."
            )
    if "replica_energies" not in wc.outputs or "replica_distances" not in wc.outputs:
        with main_out:
            print("No accepted energy/distance arrays are available in this result.")
        btn_show.disabled = False
        return

    energies_array = (
        wc.outputs["replica_energies"].get_array("energies") * helpers.HART_2_EV
    )
    distances_array = (
        wc.outputs["replica_distances"].get_array("distances") * helpers.BOHR_2_ANG
    )

    if (
        energies_array.ndim != 2
        or not len(energies_array)
        or energies_array.shape != distances_array.shape
    ):
        with main_out:
            print("No matching accepted energy/distance steps are available to plot.")
        btn_show.disabled = False
        return
    energies_array = np.array([e_arr - e_arr[0] for e_arr in energies_array])

    # Display/export copies follow the calculation model; no historical extras change.
    for rep in structure_data_list:
        the_ase, notice = structure_for_display(rep)
        all_ase.append(the_ase)
        with main_out:
            display(ipw.HTML(f"<p>Replica source {rep.pk}: {html.escape(notice)}</p>"))

    replica_html = make_replica_html(
        structure_data_list, energies_array[-1], distances_array[-1], replica_indices
    )

    barrier_list = [np.max(e_arr) for e_arr in energies_array]

    with main_out:
        f, axarr = plt.subplots(1, 2, figsize=(14, 4))

        axarr[0].plot(energies_array[-1], "o-")
        axarr[0].set_ylabel("Energy (eV)")
        axarr[0].set_xlabel("Replica nr")
        axarr[0].set_title("NEB energy profile")

        axarr[1].plot(barrier_list, "o-")
        axarr[1].axhline(barrier_list[-1], linestyle="--", color="lightgray")
        axarr[1].set_ylabel("Barrier (eV)")
        axarr[1].set_xlabel("Iteration nr")
        axarr[1].set_title("Barrier history")

        plt.show()

        display(ipw.HTML(replica_html))

        print(
            "List of all replica PKs: [{}]".format(
                ", ".join([str(replica.pk) for replica in structure_data_list])
            )
        )

    btn_show.disabled = False

In [ ]:
style = {"description_width": "120px"}
layout = {"width": "70%"}

pk_select = ipw.IntText(description="Load node: ", layout=layout, style=style)

btn_show = ipw.Button(description="Show")
btn_show.on_click(process_and_show_neb)

main_out = ipw.Output()

display(pk_select, btn_show, main_out)

In [ ]:
# Load the URL after everything is set up.
try:
    url = urllib.parse.urlsplit(jupyter_notebook_url)
    pk_select.value = urllib.parse.parse_qs(url.query)["pk"][0]
    process_and_show_neb(0)
except Exception as exc:
    print("Could not load the URL: ", exc)